<a href="https://colab.research.google.com/github/Nasfar100/Web-design/blob/main/DDoS_Detection_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Read the CSV file directly from the local path
file_path = '/content/train.csv'
df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
display(df.head())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Show dataset summary, columns, missing values, and target label distribution
print("Dataset Shape:", df.shape)
print("\n--- Missing Values Count ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n--- Target Label Distribution (Top 10) ---")
print(df['label'].value_counts(dropna=False).head(10))

In [ ]:
print("Dataset path:", file_path)
print("Dataset shape:", df.shape)

In [ ]:
print("All available labels and their counts:")

print(df["label"].value_counts(dropna=False).to_string())

In [ ]:

# Make a copy of the original dataset
df["label"] = df["label"].astype("string").str.strip()

# Select Benign and DDoS records only
condition = (
    (df["label"] == "BenignTraffic") |
    (df["label"].str.startswith("DDoS-", na=False))
)

df_binary = df.loc[condition].copy()

# Convert labels into two classes
df_binary["label"] = df_binary["label"].apply(
    lambda x: "Benign" if x == "BenignTraffic" else "DDoS"
)

# Display the new dataset information
print("Original dataset shape:", df.shape)
print("Binary dataset shape:", df_binary.shape)

print("\nNew label distribution:")
print(df_binary["label"].value_counts())


In [ ]:

# Check missing values
print("Missing values in each column:")
print(df_binary.isnull().sum().sum())

# Display columns that contain missing values
print("\nColumns with missing values:")
print(df_binary.isnull().sum()[df_binary.isnull().sum() > 0])

# Check duplicate rows
print("\nDuplicate rows:")
print(df_binary.duplicated().sum())

# Check data types
print("\nDataset information:")
df_binary.info()



In [ ]:

# Select only numeric columns
numeric_df = df_binary.select_dtypes(include=["number"])

# Check for infinite values
print("Infinite values:", np.isinf(numeric_df).sum().sum())

# Check minimum and maximum values
print("\nMinimum values:")
print(numeric_df.min())

print("\nMaximum values:")
print(numeric_df.max())



In [ ]:

# Check number of rows before removing duplicates
print("Before removing duplicates:", len(df_binary))

# Remove exact duplicate rows
df_binary = df_binary.drop_duplicates().copy()

# Check number of rows after removing duplicates
print("After removing duplicates:", len(df_binary))

# Check remaining duplicates
print("Remaining duplicates:", df_binary.duplicated().sum())



In [ ]:

constant_features = [
    col for col in df_binary.select_dtypes(include=["number"]).columns
    if df_binary[col].nunique() <= 1
]

print("Number of constant features:", len(constant_features))
print("Constant features:", constant_features)


In [ ]:

# Remove constant features
df_binary = df_binary.drop(columns=constant_features)

# Display updated dataset information
print("Updated dataset shape:", df_binary.shape)

# Display remaining columns
print("\nRemaining columns:")
print(df_binary.columns.tolist())



In [ ]:

# Separate input features (X) and target label (y)
X = df_binary.drop(columns=["label"])
y = df_binary["label"]

# Check their shapes
print("X shape:", X.shape)
print("y shape:", y.shape)

# Check target distribution
print("\nTarget distribution:")
print(y.value_counts())


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Initialize and train the Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Get feature importance from the trained model
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

# Sort features from highest to lowest importance
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

# Display the top 10 features
print("Top 10 Important Features:")
print(importance_df.head(10).to_string(index=False))

# Select the top 6 features
top_6_features = importance_df.head(6)["Feature"].tolist()

print("\nSelected 6 Features:")
print(top_6_features)

In [ ]:

# Select six important features
X_train_6 = X_train[top_6_features]
X_test_6 = X_test[top_6_features]

# Create a second Random Forest model
rf_model_6 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# Train using six features
rf_model_6.fit(X_train_6, y_train)

print("Random Forest model with 6 features trained successfully!")

print("\nTraining data shape:", X_train_6.shape)
print("Testing data shape:", X_test_6.shape)


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Predictions using all 39 features
y_pred_full = rf_model.predict(X_test)

# Predictions using 6 features
y_pred_6 = rf_model_6.predict(X_test_6)

# Function to evaluate a model
def evaluate_model(y_true, y_pred, model_name):

    print("\n", model_name)
    print("-" * 45)

    print("Accuracy:", accuracy_score(y_true, y_pred))

    print("Precision:", precision_score(
        y_true, y_pred, pos_label="DDoS"
    ))

    print("Recall:", recall_score(
        y_true, y_pred, pos_label="DDoS"
    ))

    print("F1-score:", f1_score(
        y_true, y_pred, pos_label="DDoS"
    ))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(
        y_true, y_pred,
        labels=["Benign", "DDoS"]
    ))

# Evaluate both models
evaluate_model(y_test, y_pred_full, "Random Forest - 39 Features")

evaluate_model(y_test, y_pred_6, "Random Forest - 6 Features")


In [ ]:

# Check whether any selected features are identical
# to the target label after numeric encoding.

print("Training and testing index overlap:")
print(len(set(X_train.index) & set(X_test.index)))

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))

print("\nCheck whether label is included in X:")
print("label" in X_train.columns)


In [ ]:

# Compare the average feature values
train_means = X_train.mean()
test_means = X_test.mean()

comparison = pd.DataFrame({
    "Training Mean": train_means,
    "Testing Mean": test_means
})

comparison["Difference"] = (
    comparison["Training Mean"] -
    comparison["Testing Mean"]
).abs()

print("Feature distribution comparison:")
print(comparison.sort_values(
    by="Difference",
    ascending=False
).head(10))


In [ ]:

from sklearn.metrics import classification_report

print("Detailed evaluation of 6-feature Random Forest:")

print(classification_report(
    y_test,
    y_pred_6,
    labels=["Benign", "DDoS"],
    target_names=["Normal Traffic (Benign)", "DDoS Attack"],
    digits=4
))


In [ ]:

import joblib

# Save the trained six-feature model
joblib.dump(rf_model_6, "ddos_random_forest_model.pkl")

# Save the selected feature names
joblib.dump(top_6_features, "ddos_features.pkl")

print("Model and selected features saved successfully!")


In [ ]:

import joblib

# Load the saved model and feature names
loaded_model = joblib.load("ddos_random_forest_model.pkl")
loaded_features = joblib.load("ddos_features.pkl")

print("Model loaded:", type(loaded_model))
print("Selected features:", loaded_features)
print("Number of features:", len(loaded_features))


In [ ]:

# Select one record from the testing dataset
sample = X_test[loaded_features].iloc[[0]]

# Make a prediction
prediction = loaded_model.predict(sample)

print("Input features:")
print(sample)

print("\nPredicted result:", prediction[0])
print("Actual result:", y_test.iloc[0])


In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import plotly
import reportlab

print("All libraries installed successfully!")
print("Streamlit version:", st.__version__)

In [ ]:
!pip install -q streamlit plotly reportlab joblib

In [ ]:
%%writefile /content/app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import io
import os
from pathlib import Path
from datetime import datetime

import plotly.express as px
import plotly.graph_objects as go

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak
)
from reportlab.lib.units import inch

# =====================================================
# PAGE CONFIGURATION
# =====================================================
st.set_page_config(
    page_title="DDoS Detection System",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# =====================================================
# CUSTOM CSS
# =====================================================
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
html, body, [class*="css"] {
    font-family: 'Inter', sans-serif;
}
.stApp {
    background: #0b1120;
    color: #e5e7eb;
}
[data-testid="stSidebar"] {
    background: #111827;
    border-right: 1px solid #263449;
}
[data-testid="stSidebar"] * {
    color: #e5e7eb;
}
.main-title {
    font-size: 32px;
    font-weight: 800;
    color: #f8fafc;
    margin-bottom: 5px;
}
.subtitle {
    color: #94a3b8;
    font-size: 14px;
    margin-bottom: 25px;
}
.section-title {
    font-size: 21px;
    font-weight: 700;
    color: #f8fafc;
    margin-top: 20px;
    margin-bottom: 15px;
}
.metric-card {
    background: linear-gradient(145deg, #172338, #111827);
    border: 1px solid #29384e;
    border-radius: 13px;
    padding: 20px;
    min-height: 115px;
}
.metric-label {
    color: #94a3b8;
    font-size: 13px;
    font-weight: 600;
}
.metric-value {
    color: #f8fafc;
    font-size: 29px;
    font-weight: 800;
    margin-top: 8px;
}
.metric-note {
    color: #64748b;
    font-size: 12px;
    margin-top: 5px;
}
.info-card {
    background: #172338;
    border: 1px solid #29384e;
    border-radius: 12px;
    padding: 20px;
    margin-bottom: 15px;
}
div.stButton > button {
    background: #2563eb;
    color: white;
    border: none;
    border-radius: 8px;
    padding: 10px 22px;
    font-weight: 600;
    width: 100%;
}
div.stButton > button:hover {
    background: #1d4ed8;
    color: white;
}
.stDownloadButton > button {
    background: #172338;
    border: 1px solid #3b82f6;
    color: #e5e7eb;
    border-radius: 8px;
    width: 100%;
}
hr {
    border-color: #29384e;
}
[data-testid="stDataFrame"] {
    border: 1px solid #29384e;
    border-radius: 10px;
}
</style>
""", unsafe_allow_html=True)

# =====================================================
# MODEL FILES
# =====================================================
BASE_DIR = Path("/content")
MODEL_PATH = BASE_DIR / "ddos_random_forest_model.pkl"
FEATURE_PATH = BASE_DIR / "ddos_features.pkl"

@st.cache_resource
def load_model():
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")
    if not FEATURE_PATH.exists():
        raise FileNotFoundError(f"Feature file not found: {FEATURE_PATH}")
    model = joblib.load(MODEL_PATH)
    features = joblib.load(FEATURE_PATH)
    return model, features

try:
    model, selected_features = load_model()
except Exception as e:
    st.error(f"Unable to load model: {e}")
    st.stop()

# =====================================================
# SIDEBAR
# =====================================================
with st.sidebar:
    st.markdown("# 🛡️ CyberGuard")
    st.caption("NETWORK INTRUSION DETECTION SYSTEM")
    st.divider()
    page = st.radio(
        "NAVIGATION",
        [
            "Dashboard",
            "Upload & Detection",
            "Prediction Results",
            "Analytics",
            "Export Reports",
            "About Project"
        ]
    )
    st.divider()
    st.markdown("### Model Information")
    st.write("**Algorithm:** Random Forest")
    st.write("**Classification:** Binary")
    st.write("**Classes:** Benign / DDoS")
    st.write(f"**Features:** {len(selected_features)}")
    st.divider()
    st.caption("DDoS Detection System | CICIoT2023")

# =====================================================
# SESSION STATE
# =====================================================
if "results" not in st.session_state:
    st.session_state.results = None
if "original_data" not in st.session_state:
    st.session_state.original_data = None

# =====================================================
# REUSABLE METRIC CARD
# =====================================================
def metric_card(label, value, note=""):
    st.markdown(f"""
    <div class="metric-card">
        <div class="metric-label">{label}</div>
        <div class="metric-value">{value}</div>
        <div class="metric-note">{note}</div>
    </div>
    """, unsafe_allow_html=True)

# =====================================================
# PDF REPORT GENERATOR
# =====================================================
def create_pdf_report(results):
    buffer = io.BytesIO()
    doc = SimpleDocTemplate(
        buffer,
        pagesize=A4,
        rightMargin=40,
        leftMargin=40,
        topMargin=40,
        bottomMargin=40
    )
    styles = getSampleStyleSheet()
    elements = []
    total = len(results)
    ddos_count = (results["Predicted Label"] == "DDoS").sum()
    benign_count = (results["Predicted Label"] == "Benign").sum()
    attack_percentage = ddos_count / total * 100 if total else 0
    avg_confidence = results["Confidence (%)"].mean()

    elements.append(Paragraph("<b>DDOS ATTACK DETECTION REPORT</b>", styles["Title"]))
    elements.append(Spacer(1, 15))
    elements.append(Paragraph("Random Forest Based Network Traffic Classification", styles["Normal"]))
    elements.append(Spacer(1, 15))
    elements.append(Paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", styles["Normal"]))
    elements.append(Spacer(1, 20))
    elements.append(Paragraph("<b>1. Detection Summary</b>", styles["Heading2"]))

    summary_data = [
        ["Metric", "Result"],
        ["Total Records", f"{total:,}"],
        ["Benign Records", f"{benign_count:,}"],
        ["DDoS Records", f"{ddos_count:,}"],
        ["DDoS Percentage", f"{attack_percentage:.2f}%"],
        ["Average Prediction Confidence", f"{avg_confidence:.2f}%"]
    ]
    table = Table(summary_data, colWidths=[250, 180])
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#172554")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("PADDING", (0, 0), (-1, -1), 8),
        ("BACKGROUND", (0, 1), (-1, -1), colors.whitesmoke),
    ]))
    elements.append(table)
    elements.append(Spacer(1, 25))

    elements.append(Paragraph("<b>2. Selected Model Features</b>", styles["Heading2"]))
    for feature in selected_features:
        elements.append(Paragraph(f"• {feature}", styles["Normal"]))

    elements.append(Spacer(1, 20))
    elements.append(Paragraph("<b>3. Prediction Details</b>", styles["Heading2"]))
    details = [["Record", "Prediction", "Confidence"]]
    for index, row in results.head(100).iterrows():
        details.append([
            str(index + 1),
            str(row["Predicted Label"]),
            f'{row["Confidence (%)"]:.2f}%'
        ])
    detail_table = Table(details, colWidths=[90, 160, 160], repeatRows=1)
    detail_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#172554")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("PADDING", (0, 0), (-1, -1), 6),
        ("FONTSIZE", (0, 0), (-1, -1), 8),
    ]))
    elements.append(detail_table)
    elements.append(Spacer(1, 15))
    elements.append(Paragraph("Note: The complete prediction results are available in the CSV export.", styles["Italic"]))
    doc.build(elements)
    buffer.seek(0)
    return buffer

# =====================================================
# PAGE 1: DASHBOARD
# =====================================================
if page == "Dashboard":
    st.markdown(
        '<div class="main-title">🛡️ DDoS Detection Dashboard</div>',
        unsafe_allow_html=True
    )
    st.markdown(
        '<div class="subtitle">Real-time network traffic classification using a trained Random Forest machine learning model.</div>',
        unsafe_allow_html=True
    )

    if st.session_state.results is None:
        st.info("Upload a network traffic CSV file from the 'Upload & Detection' section to begin detection.")
        col1, col2, col3 = st.columns(3)
        with col1:
            metric_card("MODEL", "Random Forest", "Machine Learning")
        with col2:
            metric_card("FEATURES", str(len(selected_features)), "Selected features")
        with col3:
            metric_card("CLASSES", "2", "Benign / DDoS")
    else:
        results = st.session_state.results
        total = len(results)
        ddos = (results["Predicted Label"] == "DDoS").sum()
        benign = (results["Predicted Label"] == "Benign").sum()
        col1, col2, col3, col4 = st.columns(4)
        with col1:
            metric_card("TOTAL RECORDS", f"{total:,}", "Analyzed traffic")
        with col2:
            metric_card("DDoS DETECTED", f"{ddos:,}", "Potential attacks")
        with col3:
            metric_card("BENIGN TRAFFIC", f"{benign:,}", "Normal traffic")
        with col4:
            metric_card("ATTACK RATE", f"{ddos / total * 100:.2f}%" if total else "0%", "Percentage of predicted DDoS")

        st.markdown('<div class="section-title">Detection Overview</div>', unsafe_allow_html=True)
        col1, col2 = st.columns(2)
        with col1:
            fig = px.pie(names=["Benign", "DDoS"], values=[benign, ddos], hole=0.55, title="Traffic Classification", color_discrete_sequence=["#22c55e", "#ef4444"])
            st.plotly_chart(fig, use_container_width=True)
        with col2:
            fig = go.Figure(go.Bar(x=["Benign", "DDoS"], y=[benign, ddos], marker_color=["#22c55e", "#ef4444"]))
            st.plotly_chart(fig, use_container_width=True)

# =====================================================
# PAGE 2: UPLOAD AND DETECTION
# =====================================================
elif page == "Upload & Detection":
    st.markdown('<div class="main-title">📂 Upload Network Dataset</div>', unsafe_allow_html=True)
    uploaded_file = st.file_uploader("Choose your network traffic CSV file", type=["csv"])

    if uploaded_file is not None:
        try:
            df = pd.read_csv(uploaded_file)
            st.success("Dataset uploaded successfully!")
            missing_features = [f for f in selected_features if f not in df.columns]
            if missing_features:
                st.error("Missing required model features: " + ", ".join(missing_features))
            else:
                if st.button("🚀 Start DDoS Detection", use_container_width=True):
                    with st.spinner("Analyzing network traffic..."):
                        X = df[selected_features].copy()
                        for col in selected_features:
                            X[col] = pd.to_numeric(X[col], errors="coerce")
                        predictions = model.predict(X)
                        probabilities = model.predict_proba(X)
                        classes = list(model.classes_)
                        ddos_index = classes.index("DDoS")
                        benign_index = classes.index("Benign")
                        output = df.copy()
                        output["Predicted Label"] = predictions
                        output["Confidence (%)"] = probabilities.max(axis=1) * 100
                        st.session_state.results = output
                        st.success("Detection completed successfully!")
                        st.rerun()
        except Exception as e:
            st.error(f"Unable to read CSV file: {str(e)}")

# =====================================================
# PAGE 3: PREDICTION RESULTS
# =====================================================
elif page == "Prediction Results":
    st.markdown('<div class="main-title">🔍 Prediction Results</div>', unsafe_allow_html=True)
    results = st.session_state.results
    if results is None:
        st.warning("Please upload a dataset and run detection first.")
    else:
        st.dataframe(results.head(100), use_container_width=True)

# =====================================================
# PAGE 4: ANALYTICS
# =====================================================
elif page == "Analytics":
    st.markdown('<div class="main-title">📊 Detection Analytics</div>', unsafe_allow_html=True)
    results = st.session_state.results
    if results is None:
        st.warning("Run detection to view analytics.")
    else:
        st.metric("Average Confidence", f'{results["Confidence (%)"].mean():.2f}%')

# =====================================================
# PAGE 5: EXPORT REPORTS
# =====================================================
elif page == "Export Reports":
    st.markdown('<div class="main-title">📥 Export Detection Results</div>', unsafe_allow_html=True)
    results = st.session_state.results
    if results is None:
        st.warning("No prediction results available.")
    else:
        csv_data = results.to_csv(index=False).encode("utf-8")
        st.download_button(label="⬇️ Download Complete CSV", data=csv_data, file_name="ddos_detection_results.csv", mime="text/csv")
        pdf_data = create_pdf_report(results)
        st.download_button(label="⬇️ Download PDF Report", data=pdf_data, file_name="ddos_detection_report.pdf", mime="application/pdf")

# =====================================================
# PAGE 6: ABOUT PROJECT
# =====================================================
elif page == "About Project":
    st.markdown('<div class="main-title">ℹ️ About the Project</div>', unsafe_allow_html=True)
    st.write("This project implements a machine learning based network traffic classification system to identify DDoS attacks.")

st.markdown("---")
st.markdown('<div style="text-align:center; color:#64748b; padding:10px;">🛡️ DDoS Detection System</div>', unsafe_allow_html=True)


In [ ]:
%%writefile /content/app.py

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /content/cloudflared.deb

!dpkg -i /content/cloudflared.deb

In [ ]:
!cloudflared --version

In [ ]:
!cat /content/streamlit.log


In [ ]:
!streamlit run /content/app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true > /content/streamlit.log 2>&1 &

In [ ]:
!sleep 5
!curl -I http://127.0.0.1:8501

In [ ]:
!cloudflared tunnel --protocol http2 --url http://127.0.0.1:8501

2026-11-25T00:12:16Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-25T00:12:16Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-25T00:12:20Z INF +--------------------------------------------------------------------------------------------+
2026-11-25T00:12:20Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-25T00:12:20Z INF |  https://puts-upgrade-innovative-destinations.trycloud